In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import HuggingFaceSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

/home/gpudual/bhklab/declan/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# KiTS

completed

In [2]:
DATASET_NAME = "KiTS"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



14:16:44 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:71


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver'
]

In [3]:
description = """
The official repository of the 2023 Kidney Tumor Segmentation Challenge
"""

In [4]:
source = HuggingFaceSource(
    file_type="nifti",
    repo_id="neheller/KiTS-Challenge-Imaging",
    description=description,
    #post_download=["unzip"]
)

In [5]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [6]:

print(indexed_datasets.source_config(DATASET_NAME))

HuggingFaceSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='huggingface',
    repo_id='neheller/KiTS-Challenge-Imaging',
    post_download=['unzip'],
    description='\nThe official repository of the 2023 Kidney Tumor Segmentation Challenge\n'
)

In [ ]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

14:17:25 [info     ] Downloading all instances from Hugging Face repository neheller/KiTS-Challenge-Imaging [imgnet] call=downloaders.download:34
2026-08-06 14:17:27,129 - HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-08-06 14:17:27,190 - HTTP Request: GET https://huggingface.co/api/datasets/neheller/KiTS-Challenge-Imaging "HTTP/1.1 200 OK"
2026-08-06 14:17:27,235 - HTTP Request: GET https://huggingface.co/api/datasets/neheller/KiTS-Challenge-Imaging/revision/main "HTTP/1.1 200 OK"
2026-08-06 14:17:27,421 - HTTP Request: GET https://huggingface.co/api/datasets/neheller/KiTS-Challenge-Imaging/tree/65f1f295873a326230153c7e1de0c7dba10f0b29?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-06 14:17:27,511 - HTTP Request: GET https://huggingface.co/api/datasets/neheller/KiTS-Challenge-Imaging/tree/65f1f295873a326230153c7e1de0c7dba10f0b29?expand=false&recursive=true&limit=1000&cursor=ZXlKbWFXeGxYMjVoYldVaU9pSnpaV2R0Wlc1MFlYUnBiMjV6TDJOaGMyVmZNREF5TUR

Download complete: : ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 94.8GB            

In [ ]:
# import zipfile
# from glob import glob
# import os

# def extract_zip(zip_path, extract_to=None):
#     """
#     Extract a zip file to a specified directory
    
#     Args:
#         zip_path: Path to the zip file
#         extract_to: Directory to extract to (default: same directory as zip)
#     """
#     if extract_to is None:
#         extract_to = os.path.dirname(zip_path)
    
#     # Create extraction directory if it doesn't exist
#     os.makedirs(extract_to, exist_ok=True)
    
#     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#         zip_ref.extractall(extract_to)
#         print(f"Extracted {zip_path} to {extract_to}")

# for zip_file in glob("temp_data/COVID-19 CT Lung/*.zip"):
#     print(zip_file)
#     # Get the base name without extension to create a folder with the same name
#     zip_name = os.path.splitext(os.path.basename(zip_file))[0]
#     extract_to = os.path.join("temp_data/COVID-19 CT Lung", zip_name)
#     extract_zip(zip_file, extract_to)

temp_data/COVID-19 CT Lung/COVID-19-CT-Seg_20cases.zip

BadZipFile: File is not a zip file

In [12]:
crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="images/case_{ImageID}.nii.gz",
    mask_name_pattern="segmentations/case_{ImageID}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()


Download complete: : ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 94.8GB            
Reconstruction complete: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 94.8GB / 94.8GB            
13:56:26 [warning  ] Some files did not match any pattern. [imgtools] unmatched_count=300 total_count=1278 example_paths=['interpolated/images/case_00000.nii.gz', 'interpolated/images/case_00001.nii.gz', 'interpolated/images/case_00002.nii.gz', 'interpolated/images/case_00003.nii.gz', 'interpolated/images/case_00004.nii.gz'] patterns_tried=['images/case_{ImageID}.nii.gz', 'segmentations/case_{ImageID}.nii.gz'] hint='Check that patterns include all path segments between ro

In [13]:
index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")

index_df



,ImageID,filepath,file_type,reference_id,reference_scan
0,0,images/case_00000.nii.gz,scan,0,NaN
1,1,images/case_00001.nii.gz,scan,1,NaN
2,2,images/case_00002.nii.gz,scan,2,NaN
3,3,images/case_00003.nii.gz,scan,3,NaN
4,4,images/case_00004.nii.gz,scan,4,NaN
...,...,...,...,...,...
973,584,segmentations/case_00584.nii.gz,mask,584,images/case_00584.nii.gz
974,585,segmentations/case_00585.nii.gz,mask,585,images/case_00585.nii.gz
975,586,segmentations/case_00586.nii.gz,mask,586,images/case_00586.nii.gz
976,587,segmentations/case_00587.nii.gz,mask,587,images/case_00587.nii.gz


In [14]:
index_df

,ImageID,filepath,file_type,reference_id,reference_scan
0,0,images/case_00000.nii.gz,scan,0,NaN
1,1,images/case_00001.nii.gz,scan,1,NaN
2,2,images/case_00002.nii.gz,scan,2,NaN
3,3,images/case_00003.nii.gz,scan,3,NaN
4,4,images/case_00004.nii.gz,scan,4,NaN
...,...,...,...,...,...
973,584,segmentations/case_00584.nii.gz,mask,584,images/case_00584.nii.gz
974,585,segmentations/case_00585.nii.gz,mask,585,images/case_00585.nii.gz
975,586,segmentations/case_00586.nii.gz,mask,586,images/case_00586.nii.gz
976,587,segmentations/case_00587.nii.gz,mask,587,images/case_00587.nii.gz


In [18]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "CT"
    else:
        return "SEG"
    

index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["BodyPartExamined"] = "KIDNEY"


index_df

,ImageID,filepath,file_type,SampleID,reference_scan,Modality,BodyPartExamined,ROINames
0,0,images/case_00000.nii.gz,scan,0,NaN,CT,KIDNEY,NaN
1,1,images/case_00001.nii.gz,scan,1,NaN,CT,KIDNEY,NaN
2,2,images/case_00002.nii.gz,scan,2,NaN,CT,KIDNEY,NaN
3,3,images/case_00003.nii.gz,scan,3,NaN,CT,KIDNEY,NaN
4,4,images/case_00004.nii.gz,scan,4,NaN,CT,KIDNEY,NaN
...,...,...,...,...,...,...,...,...
973,584,segmentations/case_00584.nii.gz,mask,584,images/case_00584.nii.gz,SEG,KIDNEY,"Kidney,Tumor,Cyst"
974,585,segmentations/case_00585.nii.gz,mask,585,images/case_00585.nii.gz,SEG,KIDNEY,"Kidney,Tumor,Cyst"
975,586,segmentations/case_00586.nii.gz,mask,586,images/case_00586.nii.gz,SEG,KIDNEY,"Kidney,Tumor,Cyst"
976,587,segmentations/case_00587.nii.gz,mask,587,images/case_00587.nii.gz,SEG,KIDNEY,"Kidney,Tumor,Cyst"


In [19]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet


index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df["SampleID"] = index_df["SampleID"].astype(str)
index_df.loc[index_df["file_type"] == "mask", "ROINames"] = "Kidney,Tumor,Cyst"
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)

,ImageID,filepath,file_type,SampleID,reference_scan,Modality,BodyPartExamined,ROINames
0,0,images/case_00000.nii.gz,scan,0,NaN,CT,KIDNEY,NaN
1,1,images/case_00001.nii.gz,scan,1,NaN,CT,KIDNEY,NaN
2,2,images/case_00002.nii.gz,scan,2,NaN,CT,KIDNEY,NaN
3,3,images/case_00003.nii.gz,scan,3,NaN,CT,KIDNEY,NaN
4,4,images/case_00004.nii.gz,scan,4,NaN,CT,KIDNEY,NaN
...,...,...,...,...,...,...,...,...
973,584,segmentations/case_00584.nii.gz,mask,584,images/case_00584.nii.gz,SEG,KIDNEY,"Kidney,Tumor,Cyst"
974,585,segmentations/case_00585.nii.gz,mask,585,images/case_00585.nii.gz,SEG,KIDNEY,"Kidney,Tumor,Cyst"
975,586,segmentations/case_00586.nii.gz,mask,586,images/case_00586.nii.gz,SEG,KIDNEY,"Kidney,Tumor,Cyst"
976,587,segmentations/case_00587.nii.gz,mask,587,images/case_00587.nii.gz,SEG,KIDNEY,"Kidney,Tumor,Cyst"
